In [2]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/HP_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)



Mounted at /content/drive


In [3]:
print(df.head())
print(df['year'].unique())
print(df.columns)
# Keep only one row per location-year (average if needed)
# Define feature columns
features = ['BSI', 'Forest', 'NBR', 'NDMI', 'NDVI']

# Now group properly
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

             system:index       BSI  Forest       NBR      NDMI      NDVI  \
0  00000000000000001d85_0 -0.229320       1  0.541297  0.271396  0.755377   
1  00000000000000001d85_0 -0.107494       1  0.354963  0.159747  0.490986   
2  00000000000000001d85_0 -0.152623       1  0.386728  0.220261  0.526673   
3  00000000000000001d85_0 -0.103897       1  0.354726  0.159483  0.492475   
4  00000000000000001d86_0 -0.236806       1  0.546247  0.280282  0.773226   

   year                                               .geo  longitude  \
0  2021  {"geodesic":false,"type":"Point","coordinates"...  77.399294   
1  2022  {"geodesic":false,"type":"Point","coordinates"...  77.399294   
2  2023  {"geodesic":false,"type":"Point","coordinates"...  77.399294   
3  2024  {"geodesic":false,"type":"Point","coordinates"...  77.399294   
4  2021  {"geodesic":false,"type":"Point","coordinates"...  77.403786   

    latitude  
0  31.304042  
1  31.304042  
2  31.304042  
3  31.304042  
4  31.304042  
[2021 20

In [4]:
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

# Make sure duplicates are removed
df_unique = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

for (lat, lon), group in df_unique.groupby(['latitude', 'longitude']):
    group = group.sort_values('year')
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

Raw input shape: (30000, 4, 5)


In [5]:

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/HP_Deforestation_Predictions.csv',
    index=False
)

print("Saved:HP_Deforestation_Predictions.csv")



Label distribution:
No deforestation: 26450
Deforestation: 3550
Train shape: (24000, 4, 5)
Test shape : (6000, 4, 5)
Scaled train shape: (24000, 4, 5)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,033 (78.25 KB)

 Trainable params: 20,033 (78.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.5272 - loss: 1.1325 - precision: 0.1906 - recall: 0.9229 - val_accuracy: 0.7853 - val_loss: 0.4720 - val_precision: 0.3469 - val_recall: 0.9225
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7804 - loss: 0.7684 - precision: 0.3407 - recall: 0.9151 - val_accuracy: 0.6710 - val_loss: 0.6277 - val_precision: 0.2638 - val_recall: 0.9944
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8170 - loss: 0.6516 - precision: 0.3869 - recall: 0.9345 - val_accuracy: 0.7762 - val_loss: 0.4554 - val_precision: 0.3451 - val_recall: 0.9930
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8521 - loss: 0.5575 - precision: 0.4412 - recall: 0.9384 - val_accuracy: 0.8027 - val_loss: 0.4252 - val_precision: 0.3745 - val_recall: 0.9958
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8752 - loss: 0.4734 - precision: 0.4860 - recall: 0.9521 - val_accuracy: 0.8918 - val_loss:

In [6]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/HP_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/HP_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/HP_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


NDVI: {'mean': np.float64(-0.21561386439703167), 'std': 0.14817937269739942, 'q1': np.float64(-0.29558312), 'q3': np.float64(-0.21068149000000005), 'lower_2std': np.float64(-0.5119726097918305), 'upper_2std': np.float64(0.08074488099776717)}
NBR : {'mean': np.float64(-0.15808990731909667), 'std': 0.07412525290025093, 'q1': np.float64(-0.1987755675), 'q3': np.float64(-0.14268617500000003), 'lower_2std': np.float64(-0.30634041311959853), 'upper_2std': np.float64(-0.009839401518594804)}
BSI : {'mean': np.float64(0.08459240817638106), 'std': 0.04547348627752776, 'q1': np.float64(0.06585158311880898), 'q3': np.float64(0.11286991074007041), 'lower_2std': np.float64(-0.00635456437867446), 'upper_2std': np.float64(0.17553938073143657)}
NDMI: {'mean': np.float64(-0.06596299754668), 'std': 0.0426210738178055, 'q1': np.float64(-0.09107644), 'q3': np.float64(-0.046775029999999995), 'lower_2std': np.float64(-0.151205145182291), 'upper_2std': np.float64(0.019279150088930996)}

Deforestation Cause Su

In [8]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Andhra Pradesh map
# AP roughly: latitude 12.6–19.9, longitude 76.8–84.8
m = folium.Map(location=[16.5, 80.6], zoom_start=7)

# Load CSV
df = pd.read_csv('/content/drive/MyDrive/HP_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


cause
Urban/Other    997
Unknown         45
Agriculture      1
Name: count, dtype: int64


In [9]:
m.save('/content/drive/MyDrive/hp_adaptive.html')


In [10]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================

df_def = pd.read_csv(
    '/content/drive/MyDrive/HP_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))


# ==============================
# LOAD CAUSE DATA
# ==============================

df_cause = pd.read_csv(
    '/content/drive/MyDrive/HP_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


# ==============================
# LOAD HIMACHAL PRADESH DISTRICTS
# ==============================

hp = gpd.read_file('/content/drive/MyDrive/HP.geojson')

# CRS SAFETY
if hp.crs is None:
    hp.set_crs("EPSG:4326", inplace=True)

hp = hp.to_crs("EPSG:4326")

# Add state label (optional)
hp["state"] = "Himachal Pradesh"

print(hp.head())


# ==============================
# MERGE CAUSE DATA
# ==============================

df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())


# ==============================
# CREATE POINT GEOMETRY
# ==============================

geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 6000
    latitude  longitude  deforestation
0  31.564553  76.945645              0
1  32.646125  76.129975              0
2  31.938252  76.968103              0
3  31.938252  76.967204              1
4  32.815906  76.003312              0
Total deforestation samples: 1043
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  31.938252  76.967204              1     0.132363   -0.012636    0.024757   
1  32.877890  76.034753              1     0.103169    0.045947    0.035978   
2  32.228408  76.385994              1    -0.044169   -0.124434    0.042244   
3  31.626537  77.406481              1    -0.258644   -0.138778    0.033127   
4  31.940049  76.975289              1     0.321081    0.206813   -0.094231   

   NDMI_change        cause  
0    -0.069201  Urban/Other  
1    -0.045191  Urban/Other  
2    -0.035486  Urban/Other  
3    -0.001648  Urban/Other  
4     0.108767  Urban/Other  
  REMARKS_2 Country        State_Name State_Code Dist_Nam

In [11]:
# ==============================
# SPATIAL JOIN (Points → Districts)
# ==============================

gdf_joined = gpd.sjoin(
    gdf_points,
    hp,                # Himachal Pradesh GeoDataFrame
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())


# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================

if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# cleanup
gdf_joined.drop(
    columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
    inplace=True
)


# ==============================
# DISTRICT SUMMARY — HP
# ==============================

district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/HP_District_Wise_Deforestation.csv',
    index=False
)

print("Saved HP district summary")


# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================

cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/HP_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved HP cause summary")

    latitude  longitude  deforestation        cause  \
0  31.938252  76.967204              1  Urban/Other   
1  32.877890  76.034753              1  Urban/Other   
2  32.228408  76.385994              1  Urban/Other   
3  31.626537  77.406481              1  Urban/Other   
4  31.940049  76.975289              1  Urban/Other   

                    geometry  index_right REMARKS_2 Country        State_Name  \
0   POINT (76.9672 31.93825)          5.0      None   India  Himachal Pradesh   
1  POINT (76.03475 32.87789)         11.0      None   India  Himachal Pradesh   
2  POINT (76.38599 32.22841)          2.0      None   India  Himachal Pradesh   
3  POINT (77.40648 31.62654)          4.0      None   India  Himachal Pradesh   
4  POINT (76.97529 31.94005)          5.0      None   India  Himachal Pradesh   

  State_Code Dist_Name Dist_Code             state  
0         02     Mandi       027  Himachal Pradesh  
1         02    Chamba       023  Himachal Pradesh  
2         02    Kangra 

In [12]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Himachal Pradesh District Boundaries
# =====================================================
hp = gpd.read_file('/content/drive/MyDrive/HP.geojson')

# CRS safety
if hp.crs is None:
    hp.set_crs(epsg=4326, inplace=True)

hp = hp.to_crs(epsg=4326)
hp["state"] = "Himachal Pradesh"
gdf_districts = hp.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/HP_Deforestation_Predictions.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# Merge
gdf_districts = gdf_districts.merge(district_total, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_deforestation, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_afforestation, on="Dist_Name", how="left")

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 7: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 8: Create Folium Map (center Himachal Pradesh)
# =====================================================
m = folium.Map(location=[31.10, 77.17], zoom_start=8)

# =====================================================
# STEP 9: Choropleth
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Himachal Pradesh)"
).add_to(m)

# =====================================================
# STEP 10: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 11: Alert Popup
# =====================================================
state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Himachal Pradesh Afforestation Alert</b><br><br>
Total Deforestation Points: <b>{int(state_def)}</b><br><br>
High Impact Districts:<br>
{top_districts_html}<br>
🌱 Immediate afforestation drives recommended.
"""

folium.Marker(
    location=[31.10, 77.17],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 12: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/hp_forest.html")

print("✅ Himachal Pradesh map saved successfully with afforestation recommendation!")

/tmp/ipykernel_5260/1541178079.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Himachal Pradesh map saved successfully with afforestation recommendation!
